In [2]:
from pybaseball import statcast
import pandas as pd
import numpy as np

In [64]:
def get_transformed_data(start_dt, end_dt):
    df = statcast(start_dt = start_dt, end_dt = end_dt).sort_values(by=['game_date', 'at_bat_number'])
    minimal_pitch_columns = [
        'pitch_type',           # What type of pitch
        'release_speed',        # How fast
        'plate_x',              # Horizontal location at plate
        'plate_z',              # Vertical location at plate
        'pfx_x',                # Horizontal movement 
        'pfx_z',                # Vertical movement
        'release_spin_rate',    # Spin rate
        'zone'                  # Strike zone location
        ]
    
    pitch_context_cols = [
    # Count situation - heavily influences pitch selection
    'balls',                    # Pre-pitch ball count
    'strikes',                  # Pre-pitch strike count
    
    # Game situation
    'inning',                   # Which inning
    'inning_topbot',            # Top or bottom of inning
    'outs_when_up',             # Number of outs
    'home_score',               # Home team score
    'away_score',               # Away team score
    'bat_score_diff',           # Batting team score differential
    
    # Baserunner situation - affects pitch strategy
    'on_1b',                    # Runner on 1st base
    'on_2b',                    # Runner on 2nd base  
    'on_3b',                    # Runner on 3rd base
    
    # Player characteristics
    'stand',                    # Batter handedness (L/R)
    'p_throws',                 # Pitcher handedness (L/R)
    'batter',                   # Batter ID (for player-specific tendencies)
    'pitcher',                  # Pitcher ID (for pitcher-specific tendencies)
    
    # Pitch sequence context
    'pitch_number',             # Pitch number in at-bat
    'at_bat_number',            # At-bat number in game
    
    # Strike zone reference
    'sz_top',                   # Top of batter's strike zone
    'sz_bot',                   # Bottom of batter's strike zone
    
    # Pitcher workload/fatigue indicators  
    'n_thruorder_pitcher',      # Times through batting order for pitcher
    'pitcher_days_since_prev_game',  # Days rest for pitcher
    
    # Batter recent activity
    'n_priorpa_thisgame_player_at_bat',  # Prior plate appearances this game
    'batter_days_since_prev_game',       # Days since batter's last game
    
    # Age factors (experience)
    'age_pit',                  # Pitcher age
    'age_bat',                  # Batter age
    
    # Defensive positioning
    'if_fielding_alignment',    # Infield alignment
    'of_fielding_alignment',    # Outfield alignment
    ]

    game_context_cols = [
    'game_type',                # Regular season, playoffs, etc.
    'home_team',                # Home team
    'away_team',                # Away team
    'game_date',                # Date of game
    ]
    
    game_context = (
    df.sort_values(['home_team', 'game_date', 'at_bat_number'])
      .groupby(['home_team', 'game_date'])[game_context_cols]
      .first()
    )
    pitch_context = df[pitch_context_cols]
    pitch = df[minimal_pitch_columns]
    play_results = df['events']
    pitch_results = df['description']
    home_team_win_target = (
        df.sort_values(['home_team', 'game_date', 'at_bat_number'])
        .groupby(['home_team', 'game_date'])['home_win_exp']
        .last()
        .pipe(np.round)
        )
    
    res_dict = {
        'game_context': game_context,
        'pitch_context': pitch_context,
        'pitch' : pitch,
        'pitch_result' : pitch_results,
        'at_bat_target' : play_results,
        'Game_target' : home_team_win_target.reset_index()['home_win_exp']
    }   

    return res_dict

In [87]:
train = get_transformed_data('2025-03-18', '2025-10-01')
test = get_transformed_data('2025-10-02', '2026-03-17')

This is a large query, it may take a moment to complete


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pybaseball/statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)
100%|██████████| 198/198 [00:20<00:00,  9.62it/s]


This is a large query, it may take a moment to complete


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pybaseball/statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)


Skipping offseason dates


100%|██████████| 48/48 [00:04<00:00, 11.72it/s]


In [103]:
train_dummies = pd.get_dummies(train['game_context'])
test_dummies = pd.get_dummies(test['game_context'])

test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

train_X = train_dummies.values
test_X = test_dummies.values

In [104]:
train_y = np.array(train['Game_target'])
test_y = np.array(test['Game_target'])

In [ ]:
from sklearn.neural_network import MLPClassifier

np.random.seed(0)
model = MLPClassifier(hidden_layer_sizes=[5, 5], early_stopping=True)

model.fit(train_X, train_y)
model.score(train_X, train_y), model.score(test_X, test_y)

(0.655515730784548, 0.6)

In [134]:
test_y.mean()

np.float64(0.6153846153846154)

1.0